In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("source_system", "")
dbutils.widgets.dropdown("source_file_system", "parquet", ["parquet", "csv"])


landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
source_system = dbutils.widgets.get("source_system")
source_file_system = dbutils.widgets.get("source_file_system")

In [0]:
path=f"/Volumes/caresyncdata/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}"


In [0]:
import os

print(f"Checking path: {path}")

if not os.path.exists(path):
    print(f"ℹ️ No new files received. Path does not exist: {path}")
    dbutils.notebook.exit("No new files received")
else:
    print(f"✅ Path found: {path} — proceeding with ingestion.")

In [0]:
if source_file_system == "csv":
    df_landing = spark.read.format("csv").option("header", True).option("inferSchema", True).load(path)
else:
    df_landing = spark.read.format("parquet").option("header", True).option("inferSchema", True).load(path)


In [0]:
from pyspark.sql.functions import *
df_update=df_landing.withColumn("landing_timestamp",lit(landing_timestamp)).withColumn("insert_timestamp",current_timestamp())
display(df_update)

In [0]:
df_update.write.mode("append").saveAsTable(f"caresyncdata.bronze.{table_name}")